# CBIS-DDSM: **Fine-tuned** Attention Localization (Grad-CAM++ @ 14x14)

Follow-up to `cbis_attention_localization.ipynb`, whose *frozen*-backbone result was a negative
(localization pointing-game ~0.5%). This version tests the two levers most likely to rescue it:

1. **Fine-tune the backbone** (two-phase: warm up the head frozen, then unfreeze `block5/6/7/top`
   at a low LR, BatchNorm kept frozen) -> features become lesion-specific.
2. **Higher-resolution Grad-CAM++** at `block5c_add` (**14x14**, vs the coarse 7x7 `block6a`) ->
   sharper localization of small lesions.

Task = **mass vs. calcification** (learnable from full mammograms). Ground truth = ROI masks.
Everything else (patient-disjoint split, uint8 arrays, RTX-only pinning) carried over from the
working notebook. Reports Grad-CAM **and** Grad-CAM++ pointing-game / IoU / Dice.


In [ ]:
# ---- GPU setup: PIN to the discrete NVIDIA GPU (hide Intel iGPU) ----
import os
os.environ.setdefault('DML_VISIBLE_DEVICES', '0')
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPUs:', gpus)
if gpus:
    try:
        tf.config.set_visible_devices([gpus[0]], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print('Pinned to:', gpus[0])
    except Exception as e:
        print('GPU config error:', e)
print('Visible GPUs:', tf.config.get_visible_devices('GPU'))

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# ============================================================ CONFIG
DATA_ROOT  = 'D:/cbis-ddsm'
JPEG_ROOT  = os.path.join(DATA_ROOT, 'jpeg')
CSV_DIR    = os.path.join(DATA_ROOT, 'csv')
DICOM_INFO = os.path.join(CSV_DIR, 'dicom_info.csv')
SAVE_DIR   = 'G:/My Drive/VS_CC/models'
os.makedirs(SAVE_DIR, exist_ok=True)

TRAIN_CSVS = ['mass_case_description_train_set.csv', 'calc_case_description_train_set.csv']
TEST_CSVS  = ['mass_case_description_test_set.csv',  'calc_case_description_test_set.csv']

TASK        = 'abnormality'          # mass vs calc (learnable) | 'pathology' (B/M)
IMG_SIZE    = 224
CAM_LAYER   = 'block5c_add'          # 14x14 (finer than block6a 7x7). Alt: 'block3b_add' (28x28)
UNFREEZE    = ('block5', 'block6', 'block7', 'top')
WARMUP_EPOCHS, FT_EPOCHS = 8, 25
LR_WARMUP, LR_FT = 1e-3, 1e-5
ROT_ANGLES  = [0]
BATCH_SIZE  = 8
RANDOM_STATE = 2021
MASK_THRESH = 127

## 1. Resolve full image + ROI mask via `dicom_info.csv` (same as before)

In [ ]:
def _to_local(p):
    p = str(p).replace('\\', '/'); i = p.find('jpeg/')
    return os.path.join(JPEG_ROOT, p[i + 5:]) if i >= 0 else p

def build_uid_maps(dicom_info):
    di = pd.read_csv(dicom_info); di['SeriesDescription'] = di['SeriesDescription'].fillna('')
    di['local'] = di['image_path'].apply(_to_local); di['uid'] = di['SeriesInstanceUID'].astype(str)
    full, mask, allf = {}, {}, {}
    for _, r in di.iterrows():
        allf.setdefault(r['uid'], []).append(r['local']); d = r['SeriesDescription'].lower()
        if 'full' in d: full.setdefault(r['uid'], r['local'])
        elif 'roi mask' in d: mask.setdefault(r['uid'], r['local'])
    return full, mask, allf

FULL_MAP, MASK_MAP, ANY_MAP = build_uid_maps(DICOM_INFO)
print('UIDs: full=%d mask=%d' % (len(FULL_MAP), len(MASK_MAP)))

def series_uid(p):
    parts = str(p).replace('\\', '/').strip().split('/'); return parts[-2] if len(parts) >= 2 else None
def is_mask_like(g):
    return g is not None and np.mean((g <= 10) | (g >= 245)) > 0.90 and len(np.unique(g)) <= 64
def resolve_mask(p):
    uid = series_uid(p)
    if uid in MASK_MAP: return MASK_MAP[uid]
    for c in ANY_MAP.get(uid, []):
        if is_mask_like(cv2.imread(c, cv2.IMREAD_GRAYSCALE)): return c
    return None

def load_tables(names):
    df = pd.concat([pd.read_csv(os.path.join(CSV_DIR, n)) for n in names if os.path.exists(os.path.join(CSV_DIR, n))],
                   ignore_index=True)
    df.columns = [c.strip().replace(' ', '_').lower() for c in df.columns]; return df
def prep(df):
    if TASK == 'pathology':
        df['label'] = (df['pathology'].astype(str).str.upper().str.strip() == 'MALIGNANT').astype(int)
    else:
        src = df['abnormality_type'] if 'abnormality_type' in df.columns else df['abnormality']
        df['label'] = src.astype(str).str.upper().str.contains('MASS').astype(int)
    df['full_path'] = df['image_file_path'].apply(lambda p: FULL_MAP.get(series_uid(p)))
    df['mask_path'] = df['roi_mask_file_path'].apply(resolve_mask)
    df['patient_id'] = df['patient_id'].astype(str); df['image_key'] = df['image_file_path'].astype(str)
    return df[['patient_id', 'image_key', 'full_path', 'mask_path', 'label']].dropna(subset=['full_path'])

train_df = prep(load_tables(TRAIN_CSVS)); test_df = prep(load_tables(TEST_CSVS))
overlap = set(train_df.patient_id) & set(test_df.patient_id)
if overlap:
    train_df = train_df[~train_df.patient_id.isin(overlap)].reset_index(drop=True)
    print(f'Removed {len(overlap)} overlapping patients from train.')
print('Train:', len(train_df), '| Test:', len(test_df), '| disjoint:',
      set(train_df.patient_id).isdisjoint(set(test_df.patient_id)))

## 2. Preprocess (uint8) + mask + build arrays (augment train only)

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def preprocess_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    return np.stack([_clahe.apply(cv2.resize(img, (IMG_SIZE, IMG_SIZE)))] * 3, axis=-1)  # uint8
def load_mask(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE) if path else None
    if not is_mask_like(m): return None
    return (cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST) >= MASK_THRESH).astype(np.uint8)
def rotate(img, a):
    if a == 0: return img
    return cv2.rotate(img, {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}[a])

def build_arrays(df, augment):
    angles = ROT_ANGLES if augment else [0]
    n = len(df) * len(angles)
    X = np.empty((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8); Y = np.empty(n, dtype=np.int64)
    K = np.empty(n, dtype=object); j = 0
    for row in df.itertuples(index=False):
        base = preprocess_image(row.full_path)
        if base is None: continue
        for a in angles:
            X[j] = rotate(base, a); Y[j] = row.label; K[j] = row.image_key; j += 1
    return X[:j], Y[:j], K[:j]

x_train, y_train, k_train = build_arrays(train_df, augment=True)
x_test,  y_test,  _       = build_arrays(test_df,  augment=False)
print('x_train', x_train.shape, np.bincount(y_train), '| x_test', x_test.shape, np.bincount(y_test))
gss = GroupShuffleSplit(1, test_size=0.15, random_state=RANDOM_STATE)
tr, va = next(gss.split(x_train, y_train, groups=k_train))
xt, yt, xv, yv = x_train[tr], y_train[tr], x_train[va], y_train[va]
import gc; del x_train, y_train, k_train; gc.collect()
print('fit/val:', xt.shape, xv.shape)

### 2b. DIAGNOSTIC — image<->mask alignment

In [ ]:
probe = test_df[test_df['mask_path'].notna()].drop_duplicates('image_key').head(4)
fig, ax = plt.subplots(1, len(probe), figsize=(3*len(probe), 3.2), dpi=110); ax = np.atleast_1d(ax)
for a, (_, r) in zip(ax, probe.iterrows()):
    im = preprocess_image(r['full_path']); mk = load_mask(r['mask_path'])
    if im is not None: a.imshow(im[..., 0]/255., cmap='gray')
    if mk is not None:
        for c in cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]:
            c = c.squeeze(1)
            if c.ndim == 2: a.plot(np.append(c[:,0], c[0,0]), np.append(c[:,1], c[0,1]), color='lime', lw=1.3)
    a.axis('off')
plt.tight_layout(); plt.show()

## 3. Model: EfficientNetB0 + CBAM (fine-tunable)

In [ ]:
def cbam_block(x, ratio=16, name='cbam'):
    f = x.shape[-1]
    s1 = layers.Dense(max(1, f//ratio), activation='relu', kernel_initializer='he_normal')
    s2 = layers.Dense(f, kernel_initializer='he_normal')
    avg = layers.Reshape((1,1,f))(layers.GlobalAveragePooling2D()(x))
    mx  = layers.Reshape((1,1,f))(layers.GlobalMaxPooling2D()(x))
    ch  = layers.Activation('sigmoid')(layers.Add()([s2(s1(avg)), s2(s1(mx))]))
    x   = layers.Multiply()([x, ch])
    ap  = layers.Lambda(lambda t: tf.reduce_mean(t, -1, keepdims=True))(x)
    mp  = layers.Lambda(lambda t: tf.reduce_max(t, -1, keepdims=True))(x)
    sp  = layers.Conv2D(1, 7, padding='same', activation='sigmoid', name=name+'_spatial')(layers.Concatenate()([ap, mp]))
    return layers.Multiply()([x, sp])

def build_model(lr):
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = cbam_block(base.output)                              # on top_activation (7x7x1280)
    x = layers.Activation('linear', name='loc_features')(x)
    g = layers.Dropout(0.5)(layers.GlobalAveragePooling2D(name='gap')(x))
    out = layers.Dense(1, activation='sigmoid')(g)
    m = Model(base.input, out)
    m.compile(Adam(lr), 'binary_crossentropy', ['accuracy', tf.keras.metrics.AUC(name='auc')])
    return m, base

def unfreeze(model, base, lr):
    base.trainable = True
    for l in base.layers:                                   # unfreeze chosen blocks; keep BatchNorm frozen
        l.trainable = l.name.startswith(UNFREEZE) and not isinstance(l, layers.BatchNormalization)
    model.compile(Adam(lr), 'binary_crossentropy', ['accuracy', tf.keras.metrics.AUC(name='auc')])
    print('trainable layers after unfreeze:', sum(l.trainable for l in model.layers))

model, base = build_model(LR_WARMUP)
print('CAM layer', CAM_LAYER, model.get_layer(CAM_LAYER).output.shape)

## 4. Two-phase training (warm up frozen -> fine-tune top blocks)

In [ ]:
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=yt)
class_weight = {0: cw[0], 1: cw[1]}

print('=== Phase 1: warm up head (backbone frozen) ===')
model.fit(xt, yt, validation_data=(xv, yv), epochs=WARMUP_EPOCHS, batch_size=BATCH_SIZE,
          class_weight=class_weight,
          callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=4, restore_best_weights=True)], verbose=2)

print('\n=== Phase 2: fine-tune %s (LR=%g) ===' % (UNFREEZE, LR_FT))
unfreeze(model, base, LR_FT)
model.fit(xt, yt, validation_data=(xv, yv), epochs=FT_EPOCHS, batch_size=BATCH_SIZE,
          class_weight=class_weight,
          callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=6, restore_best_weights=True)], verbose=2)

## 5. Classification on the patient-disjoint test set

In [ ]:
def auc_ci(yt_, yp_, n=2000, seed=42):
    rng = np.random.RandomState(seed); yt_ = np.asarray(yt_); yp_ = np.asarray(yp_); s = []
    for _ in range(n):
        idx = rng.randint(0, len(yt_), len(yt_))
        if len(np.unique(yt_[idx])) == 2: s.append(roc_auc_score(yt_[idx], yp_[idx]))
    return np.percentile(s, [2.5, 97.5])

prob = model.predict(x_test, batch_size=BATCH_SIZE, verbose=0).ravel(); pred = (prob >= 0.5).astype(int)
cm = confusion_matrix(y_test, pred); tn, fp, fn, tp = cm.ravel()
cnn_auc = roc_auc_score(y_test, prob); lo, hi = auc_ci(y_test, prob)
pos, neg = ('Mass', 'Calc') if TASK == 'abnormality' else ('Malignant', 'Benign')
print(f'Accuracy {accuracy_score(y_test, pred):.4f} | ROC-AUC {cnn_auc:.4f} (95% CI {lo:.3f}-{hi:.3f})')
print(f'Sensitivity({pos}) {tp/(tp+fn) if tp+fn else 0:.3f} | Specificity({neg}) {tn/(tn+fp) if tn+fp else 0:.3f}')
print(classification_report(y_test, pred, target_names=[neg, pos]))

## 6. Localization: **Grad-CAM vs Grad-CAM++** @ `CAM_LAYER` (14x14) vs ROI masks

In [ ]:
def _cam_model(layer):
    return Model(model.inputs, [model.get_layer(layer).output, model.output])

def gradcam(arr, layer=CAM_LAYER):
    gm = _cam_model(layer)
    with tf.GradientTape() as t:
        conv, pred = gm(arr); y = pred[:, 0]
    grads = t.gradient(y, conv)
    w = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, -1))
    return (cam / (tf.reduce_max(cam) + 1e-8)).numpy()

def gradcam_pp(arr, layer=CAM_LAYER):
    gm = _cam_model(layer)
    with tf.GradientTape() as t:
        conv, pred = gm(arr); y = pred[:, 0]
    grads = t.gradient(y, conv)[0]                          # (H,W,C)
    g2, g3 = grads ** 2, grads ** 3
    denom = 2 * g2 + tf.reduce_sum(conv[0], axis=(0, 1)) * g3
    alpha = g2 / tf.where(denom != 0, denom, tf.ones_like(denom))
    w = tf.reduce_sum(alpha * tf.nn.relu(grads), axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, -1))
    return (cam / (tf.reduce_max(cam) + 1e-8)).numpy()

def localization(fn, verbose=True, tag=''):
    pg, ious, dices, dists = [], [], [], []; diag = IMG_SIZE * np.sqrt(2)
    for key, rows in test_df[test_df['mask_path'].notna()].groupby('image_key'):
        base_img = preprocess_image(rows.iloc[0]['full_path'])
        if base_img is None: continue
        gt = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)
        for _, r in rows.iterrows():
            m = load_mask(r['mask_path'])
            if m is not None: gt |= m
        if gt.sum() == 0: continue
        heat = cv2.resize(fn(base_img[None, ...]).astype(np.float32), (IMG_SIZE, IMG_SIZE))
        py, px = np.unravel_index(int(np.argmax(heat)), heat.shape)
        pg.append(int(gt[py, px] == 1))
        pm = (heat >= 0.5 * heat.max()).astype(np.uint8)
        inter = np.logical_and(pm, gt).sum(); union = np.logical_or(pm, gt).sum()
        ious.append(inter / union if union else 0.0)
        dices.append(2 * inter / (pm.sum() + gt.sum()) if (pm.sum() + gt.sum()) else 0.0)
        ys, xs = np.where(gt); dists.append(np.hypot(px - xs.mean(), py - ys.mean()) / diag)
    res = dict(n=len(pg), pointing_game=float(np.mean(pg)), iou=float(np.mean(ious)),
               dice=float(np.mean(dices)), centroid_dist=float(np.mean(dists)))
    if verbose:
        print(f"[{tag}] n={res['n']} pointing-game={res['pointing_game']:.3f} "
              f"IoU={res['iou']:.3f} Dice={res['dice']:.3f} centroid-dist={res['centroid_dist']:.3f}")
    return res

loc_gc  = localization(gradcam,    tag='Grad-CAM   @'+CAM_LAYER)
loc_gcpp = localization(gradcam_pp, tag='Grad-CAM++ @'+CAM_LAYER)

### 6b. Overlays (image + GT mask + Grad-CAM++)

In [ ]:
ov = test_df[test_df['mask_path'].notna()].drop_duplicates('image_key').head(6)
fig, ax = plt.subplots(1, len(ov), figsize=(3*len(ov), 3.2), dpi=110); ax = np.atleast_1d(ax)
for a, (_, r) in zip(ax, ov.iterrows()):
    im = preprocess_image(r['full_path']); gt = load_mask(r['mask_path'])
    if im is None: a.axis('off'); continue
    heat = cv2.resize(gradcam_pp(im[None, ...]), (IMG_SIZE, IMG_SIZE))
    a.imshow(im[..., 0]/255., cmap='gray'); a.imshow(heat, cmap='jet', alpha=0.4)
    if gt is not None:
        for c in cv2.findContours(gt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]:
            c = c.squeeze(1)
            if c.ndim == 2: a.plot(np.append(c[:,0], c[0,0]), np.append(c[:,1], c[0,1]), color='lime', lw=1.3)
    a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'cbis_ft_localization_overlays.png'), dpi=200, bbox_inches='tight'); plt.show()

## 7. Summary + save

In [ ]:
best = loc_gcpp if loc_gcpp['pointing_game'] >= loc_gc['pointing_game'] else loc_gc
summary = (
    f"Fine-tuned EfficientNetB0+CBAM (two-phase; unfroze {UNFREEZE}) on CBIS-DDSM full mammograms "
    f"for mass vs. calcification (patient-disjoint split). Test ROC-AUC {cnn_auc:.3f} "
    f"(95% CI {lo:.3f}-{hi:.3f}). Weakly-supervised localization at {CAM_LAYER} (14x14): "
    f"Grad-CAM pointing-game {loc_gc['pointing_game']*100:.1f}% / IoU {loc_gc['iou']:.3f}; "
    f"Grad-CAM++ pointing-game {loc_gcpp['pointing_game']*100:.1f}% / IoU {loc_gcpp['iou']:.3f} / "
    f"Dice {loc_gcpp['dice']:.3f} over {loc_gcpp['n']} masked test mammograms, vs ROI-mask GT. "
    f"(Frozen-backbone baseline was pointing-game ~0.5%.)"
)
print(summary)
open(os.path.join(SAVE_DIR, 'cbis_ft_localization_summary.txt'), 'w').write(summary)
model.save_weights(os.path.join(SAVE_DIR, 'efficientnet_cbam_cbis_finetuned.weights.h5'))
print('\nSaved weights + summary to', SAVE_DIR)